In [1]:
!pip install fastapi uvicorn pyngrok nest-asyncio

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-7B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [3]:
import asyncio, json, re, logging, time

class LLM:
    def __init__(self, model="huggingface"):
        self.model = model

    def complete(self, messages):
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=2048
        )
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return re.sub(r'<\|im_end\|>', '', response)




In [4]:
llm = LLM(model)

from pydantic import BaseModel
from typing import List, Optional, Literal
from datetime import datetime

class Message(BaseModel):
    role: Literal['system', 'user']  # roles comunes en chats
    content: str

class ChatRequest(BaseModel):
    messages: List[Message] = []  # Valor por defecto lista vacía



In [5]:
from fastapi import FastAPI, Request
from pyngrok import ngrok
import nest_asyncio
import uvicorn

ngrok.set_auth_token("344HT0PzWr1pGVLwZBa7KWXfxXE_4FMsfMKfHFpG8ZAQXrpS7")


nest_asyncio.apply()  # Permite correr uvicorn en el loop de Colab

app = FastAPI()


@app.get("/")
def home():
    return {"message": "Hola desde Colab + FastAPI!"}



@app.post("/chat")
async def chat(body: ChatRequest):

    response = llm.complete(body.messages)


    return {
        "response": response
    }

In [ ]:
# Crear túnel en el puerto 8000
public_url = ngrok.connect(8000)
print("URL pública:", public_url)

config = uvicorn.Config(app=app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

await server.serve()


URL pública: NgrokTunnel: "https://transmarginally-unrebuffed-else.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [8303]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2803:9800:9885:a4dc:a535:bf83:3e00:5fdf:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:a535:bf83:3e00:5fdf:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:a535:bf83:3e00:5fdf:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:a535:bf83:3e00:5fdf:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:a535:bf83:3e00:5fdf:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:a535:bf83:3e00:5fdf:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:a535:bf83:3e00:5fdf:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:a535:bf83:3e00:5fdf:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:a535:bf83:3e00:5fdf:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:a535:bf83:3e00:5fdf:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:a535:bf83:3e00:5fdf:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:a535:bf83:3e00:5fdf:0 - "POST /chat HTTP/1.1" 200 OK
INFO